# Phase 3: Deep Learning Method — Four Specialized U-Nets for Super-Resolution

In this notebook we train **one dedicated U-Net per degradation scenario** (4 models total).
Each network is optimized for a single *(scale factor, noise level)* combination, reducing the well-known data-dependency limitation of end-to-end deep learning methods.

| Model | Scenario | Weights file |
|---|---|---|
| U-Net 1 | s=2, σ=0.005 | `unet_s2_sig0005_weights.pth` |
| U-Net 2 | s=2, σ=0.01  | `unet_s2_sig001_weights.pth`  |
| U-Net 3 | s=4, σ=0.005 | `unet_s4_sig0005_weights.pth` |
| U-Net 4 | s=4, σ=0.01  | `unet_s4_sig001_weights.pth`  |

**Data pipeline (identical to Phase 2 / FISTA):**
$$x \;\xrightarrow{K + \text{noise}}\; y \;\xrightarrow{\uparrow_{\text{bilinear}}}\; \tilde{y}_{256} \;\xrightarrow{f_\theta}\; \hat{x}$$

The degradation is applied **on-the-fly** from the same `.pt` ground-truth dataset used by FISTA.

## 3.1 Environment Setup and Drive Mount

In [ ]:
# --- CONFIGURATION ---
BASE_DRIVE_FOLDER = 'CI_Project'
# ----------------------

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_PATH = os.path.join('/content/drive/MyDrive', BASE_DRIVE_FOLDER)
else:
    DRIVE_PATH = './CI_Project'

DATASET_PATH = os.path.join(DRIVE_PATH, 'dataset')
WEIGHTS_PATH = os.path.join(DRIVE_PATH, 'weights')
OUTPUT_PATH  = os.path.join(DRIVE_PATH, 'outputs', 'unet')

os.makedirs(WEIGHTS_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH,  exist_ok=True)

print(f"Drive path  : {DRIVE_PATH}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Weights path: {WEIGHTS_PATH}")
print(f"Output path : {OUTPUT_PATH}")

In [ ]:
import sys

# Clone or update the IPPy fork
if not os.path.exists('IPPy'):
    !git clone https://github.com/alessandrocampedelli/IPPy.git
else:
    %cd IPPy
    !git pull
    %cd ..

# Install extra dependencies
!pip install -q PyWavelets astra-toolbox

# Add IPPy to the Python path
sys.path.insert(0, os.path.abspath('IPPy'))

# Verify import
from IPPy import operators
from IPPy.nn import models
from IPPy.utilities import metrics
from IPPy.utilities._utilities import gaussian_noise
print("IPPy library imported successfully!")

## 3.2 Hyperparameters

Global training settings shared by all four models. Each scenario produces its own weights and checkpoint files.

In [ ]:
import torch

# ─── Scenarios — one U-Net will be trained per entry ────────────────────────
ALL_SCENARIOS = [
    (2, 0.005),   # s=2, σ=0.005
    (2, 0.01),    # s=2, σ=0.01
    (4, 0.005),   # s=4, σ=0.005
    (4, 0.01),    # s=4, σ=0.01
]

# ─── Dataset ────────────────────────────────────────────────────────────────
IMG_SHAPE  = (256, 256)

# ─── U-Net architecture (shared across all 4 models) ────────────────────────
MIDDLE_CH  = [32, 64, 128, 256]   # Channel progression — 4 encoder levels
N_LAYERS   = 2                    # Conv layers per block
FINAL_ACT  = 'sigmoid'            # Ensures output in [0, 1]

# ─── Training ───────────────────────────────────────────────────────────────
N_EPOCHS   = 20
BATCH_SIZE = 8
LR         = 5e-4

# ─── Early Stopping ─────────────────────────────────────────────────────────
ES_PATIENCE = 7         # Stop if val loss does not improve for ES_PATIENCE epochs
ES_DELTA    = 1e-5      # Minimum improvement to be considered significant

# ─── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def scenario_tag(s, sigma):
    """Canonical tag used in file names, e.g. 's2_sig001'."""
    return f's{s}_sig{str(sigma).replace(".", "")}'

def weights_file(s, sigma):
    return os.path.join(WEIGHTS_PATH, f'unet_{scenario_tag(s, sigma)}_weights.pth')

def checkpoint_file(s, sigma):
    return os.path.join(WEIGHTS_PATH, f'unet_{scenario_tag(s, sigma)}_ckpt.pth')

print(f"Device   : {DEVICE}")
print(f"Epochs   : {N_EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}")
print(f"Early stop: patience={ES_PATIENCE}")
print(f"\nScenarios to train ({len(ALL_SCENARIOS)} models):")
for s, sigma in ALL_SCENARIOS:
    tag = scenario_tag(s, sigma)
    print(f"  s={s}, σ={sigma}  →  {weights_file(s, sigma).split('/')[-1]}")

## 3.3 On-the-Fly Dataset

The `DegradedDataset` loads HR `.pt` tensors and applies the forward model on-the-fly:
`x → y = K(x) + noise → y_up = bilinear_upsample(y, 256×256)`

This mirrors exactly the data pipeline used in Phase 2 (FISTA), ensuring a fair comparison.

In [ ]:
import glob as glob_module
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPPy.operators import DownScaling


class DegradedDataset(Dataset):
    """
    Loads HR tensors from .pt files and applies the forward model on-the-fly.

    Returns:
        (y_up, x_true) where:
          y_up   : degraded image bilinearly upsampled to IMG_SHAPE  — network input
          x_true : original HR image                                  — network target
    """
    def __init__(self, data_dir: str, scale_factor: int, noise_level: float,
                 img_shape=(256, 256), augment: bool = False):
        self.files       = sorted(glob_module.glob(os.path.join(data_dir, '*.pt')))
        assert len(self.files) > 0, f"No .pt files found in {data_dir}"
        self.K           = DownScaling(img_shape=img_shape,
                                       downscale_factor=scale_factor, mode='avg')
        self.noise_level = noise_level
        self.img_shape   = img_shape
        self.augment     = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x_true  = torch.load(self.files[idx], weights_only=True)   # (1, 256, 256)

        # Optional horizontal flip for data augmentation (training only)
        if self.augment and torch.rand(1).item() > 0.5:
            x_true = torch.flip(x_true, dims=[-1])

        # Forward model: y = K(x) + noise  →  shape (1, 1, H/s, W/s)
        x_batch = x_true.unsqueeze(0)
        y_clean = self.K(x_batch)
        y_noisy = y_clean + gaussian_noise(y_clean, self.noise_level)

        # Bilinear upsample to original resolution  →  (1, 256, 256)
        y_up = F.interpolate(y_noisy, size=self.img_shape,
                             mode='bilinear', align_corners=False)
        y_up = y_up.squeeze(0).clamp(0.0, 1.0)

        return y_up, x_true


def make_loaders(s, sigma):
    """Build train/val DataLoaders for a given scenario."""
    train_ds = DegradedDataset(os.path.join(DATASET_PATH, 'train'), s, sigma, augment=True)
    val_ds   = DegradedDataset(os.path.join(DATASET_PATH, 'val'),   s, sigma, augment=False)
    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_ld, val_ld


# Quick sanity check on the first scenario
s0, sigma0 = ALL_SCENARIOS[0]
_tl, _ = make_loaders(s0, sigma0)
y_s, x_s = next(iter(_tl))
print(f"Dataset sanity check  (s={s0}, σ={sigma0}):")
print(f"  Input  shape : {y_s.shape}  range [{y_s.min():.3f}, {y_s.max():.3f}]")
print(f"  Target shape : {x_s.shape}  range [{x_s.min():.3f}, {x_s.max():.3f}]")

## 3.4 Model Factory

A helper function that instantiates a **fresh** U-Net. Called once per scenario inside the training loop.

In [ ]:
import torch.nn as nn
from IPPy.nn.models import UNet

def build_model():
    """Instantiate a fresh U-Net and move it to DEVICE."""
    m = UNet(
        ch_in              = 1,
        ch_out             = 1,
        middle_ch          = MIDDLE_CH,
        n_layers_per_block = N_LAYERS,
        down_layers        = ('ResDownBlock',) * (len(MIDDLE_CH) - 1),
        up_layers          = ('ResUpBlock',)   * (len(MIDDLE_CH) - 1),
        final_activation   = FINAL_ACT,
    ).to(DEVICE)
    return m

# Print architecture info once
_tmp = build_model()
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
del _tmp
print(f"U-Net architecture:")
print(f"  Trainable parameters : {n_params:,}")
print(f"  Channel progression  : {MIDDLE_CH}")
print(f"  Encoder/decoder depth: {len(MIDDLE_CH)-1} levels")
print(f"  Final activation     : {FINAL_ACT}")
print(f"  Device               : {DEVICE}")

## 3.5 Training Loop — 4 Models

For each scenario the cell:
1. Checks whether a **completed checkpoint** exists → skips training
2. Checks whether a **partial checkpoint** exists → resumes from the last saved epoch
3. Otherwise starts from scratch

**Optimiser**: Adam — **Scheduler**: Cosine Annealing — **Early Stopping**: `ES_PATIENCE` epochs without improvement

In [ ]:
import time
import copy
from tqdm.auto import tqdm

all_history = {}   # all_history[(s, sigma)] = {'train_loss': [...], ...}

for s, sigma in ALL_SCENARIOS:
    tag    = scenario_tag(s, sigma)
    w_file = weights_file(s, sigma)
    c_file = checkpoint_file(s, sigma)

    print(f"\n{'='*62}")
    print(f"  Training U-Net for  s={s}, σ={sigma}  [{tag}]")
    print(f"{'='*62}")

    # ── Determine training status ────────────────────────────────────────────
    ckpt = torch.load(c_file, map_location=DEVICE, weights_only=False) \
           if os.path.exists(c_file) else None

    if ckpt is not None and ckpt.get('completed', False):
        print(f"  ✅ Already trained ({ckpt['epoch']}/{N_EPOCHS} epochs, "
              f"best val_loss={ckpt['best_val_loss']:.5f}). Loading weights...")
        model = build_model()
        model.load_state_dict(
            torch.load(w_file, map_location=DEVICE, weights_only=True)
        )
        model.eval()
        all_history[(s, sigma)] = ckpt['history']
        continue   # skip to next scenario

    model     = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=N_EPOCHS, eta_min=1e-5
    )
    loss_fn   = nn.MSELoss()

    if ckpt is not None:
        # Resume from interrupted checkpoint
        START_EPOCH   = ckpt['epoch'] + 1
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        history       = ckpt['history']
        best_val_loss = ckpt['best_val_loss']
        best_weights  = ckpt['best_model_state_dict']
        es_counter    = ckpt['es_counter']
        print(f"  🔄 Resuming from epoch {START_EPOCH}/{N_EPOCHS} "
              f"(best val_loss so far: {best_val_loss:.5f})")
    else:
        START_EPOCH   = 1
        history       = {'train_loss': [], 'val_loss': [], 'val_psnr': [], 'val_ssim': []}
        best_val_loss = float('inf')
        best_weights  = copy.deepcopy(model.state_dict())
        es_counter    = 0
        print(f"  🆕 Training from scratch for {N_EPOCHS} epochs.")

    train_loader, val_loader = make_loaders(s, sigma)
    stopped_epoch = N_EPOCHS
    print()

    for epoch in range(START_EPOCH, N_EPOCHS + 1):
        # ── Training ──────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        t0 = time.time()
        for y_batch, x_batch in tqdm(
            train_loader, desc=f"Epoch {epoch}/{N_EPOCHS} [Train]", leave=False
        ):
            y_batch = y_batch.to(DEVICE)
            x_batch = x_batch.to(DEVICE)
            optimizer.zero_grad()
            pred = model(y_batch)
            loss = loss_fn(pred, x_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        scheduler.step()

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        val_loss  = 0.0
        val_psnr  = 0.0
        val_ssim  = 0.0
        n_samples = 0
        with torch.no_grad():
            for y_batch, x_batch in tqdm(
                val_loader, desc=f"Epoch {epoch}/{N_EPOCHS} [Val]", leave=False
            ):
                y_batch = y_batch.to(DEVICE)
                x_batch = x_batch.to(DEVICE)
                pred = model(y_batch)
                val_loss += loss_fn(pred, x_batch).item()
                for i in range(pred.shape[0]):
                    p_i = pred[i:i+1].cpu()
                    x_i = x_batch[i:i+1].cpu()
                    val_psnr += metrics.PSNR(p_i, x_i)
                    val_ssim += metrics.SSIM(p_i, x_i)
                n_samples += pred.shape[0]

        val_loss /= len(val_loader)
        val_psnr /= n_samples
        val_ssim /= n_samples

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_psnr'].append(val_psnr)
        history['val_ssim'].append(val_ssim)

        elapsed = time.time() - t0
        print(f"  [{epoch:2d}/{N_EPOCHS}]  "
              f"train={train_loss:.5f}  val={val_loss:.5f}  "
              f"PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  "
              f"lr={scheduler.get_last_lr()[0]:.2e}  ({elapsed:.1f}s)")

        # ── Early stopping check ───────────────────────────────────────────────
        if val_loss < best_val_loss - ES_DELTA:
            best_val_loss = val_loss
            best_weights  = copy.deepcopy(model.state_dict())
            es_counter    = 0
        else:
            es_counter += 1
            if es_counter >= ES_PATIENCE:
                stopped_epoch = epoch
                print(f"  🛑 Early stopping at epoch {epoch} "
                      f"(no improvement for {ES_PATIENCE} epochs).")
                break

        # ── Save checkpoint after every epoch ─────────────────────────────────
        is_done = (epoch == N_EPOCHS) or (es_counter >= ES_PATIENCE)
        torch.save({
            'epoch':                 epoch,
            'completed':             is_done,
            'model_state_dict':      model.state_dict(),
            'best_model_state_dict': best_weights,
            'optimizer_state_dict':  optimizer.state_dict(),
            'scheduler_state_dict':  scheduler.state_dict(),
            'best_val_loss':         best_val_loss,
            'es_counter':            es_counter,
            'history':               history,
        }, c_file)

    # ── Restore best weights and save final model ──────────────────────────────
    model.load_state_dict(best_weights)
    model.eval()
    torch.save(best_weights, w_file)
    best_epoch = stopped_epoch - es_counter if es_counter >= ES_PATIENCE else stopped_epoch
    print(f"  ✅ Done. Best val_loss={best_val_loss:.5f} (epoch {best_epoch})")
    print(f"     Weights   → {w_file}")
    print(f"     Checkpoint→ {c_file}")

    all_history[(s, sigma)] = history

print(f"\n{'='*62}")
print("  All 4 U-Net models trained.")
print(f"{'='*62}")

## 3.6 Training Curves

One row per scenario.

In [ ]:
import matplotlib.pyplot as plt

n_rows = len(ALL_SCENARIOS)
fig, axes = plt.subplots(n_rows, 3, figsize=(16, 4 * n_rows))

for row, (s, sigma) in enumerate(ALL_SCENARIOS):
    hist = all_history.get((s, sigma))
    if hist is None or not hist['train_loss']:
        for ax in axes[row]: ax.set_visible(False)
        continue

    ep = range(1, len(hist['train_loss']) + 1)
    label = f's={s}, σ={sigma}'

    axes[row, 0].plot(ep, hist['train_loss'], label='Train', color='royalblue')
    axes[row, 0].plot(ep, hist['val_loss'],   label='Val',   color='tomato')
    axes[row, 0].set_ylabel('MSE Loss'); axes[row, 0].set_title(f'[{label}]  Loss')
    axes[row, 0].legend(); axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].plot(ep, hist['val_psnr'], color='seagreen', linewidth=1.8)
    axes[row, 1].set_ylabel('PSNR (dB)'); axes[row, 1].set_title(f'[{label}]  Val PSNR')
    axes[row, 1].grid(True, alpha=0.3)

    axes[row, 2].plot(ep, hist['val_ssim'], color='darkorange', linewidth=1.8)
    axes[row, 2].set_ylabel('SSIM'); axes[row, 2].set_title(f'[{label}]  Val SSIM')
    axes[row, 2].grid(True, alpha=0.3)

    for ax in axes[row]: ax.set_xlabel('Epoch')

plt.suptitle('U-Net Training History — all 4 scenarios', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'unet_training_history_all.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to {OUTPUT_PATH}")

## 3.7 Quantitative Evaluation on Test Set

Each U-Net is evaluated **on its own matching scenario** (the one it was trained on). This gives the fairest comparison with FISTA.

In [ ]:
unet_results = {}   # unet_results[(s, sigma)] = {'psnr': [], 'ssim': [], 'x_hat': [], ...}

for s, sigma in ALL_SCENARIOS:
    tag    = scenario_tag(s, sigma)
    w_file = weights_file(s, sigma)

    print(f"\n{'='*60}")
    print(f"  Evaluating  s={s}, σ={sigma}  [{tag}]")
    print(f"{'='*60}")

    # Load the model trained for this exact scenario
    model = build_model()
    model.load_state_dict(torch.load(w_file, map_location=DEVICE, weights_only=True))
    model.eval()

    test_ds = DegradedDataset(
        os.path.join(DATASET_PATH, 'test'), s, sigma, augment=False
    )
    unet_results[(s, sigma)] = {'psnr': [], 'ssim': [], 'x_hat': []}

    with torch.no_grad():
        for idx in range(len(test_ds)):
            y_up, x_true = test_ds[idx]              # (1, H, W)
            y_up_b   = y_up.unsqueeze(0).to(DEVICE)  # (1, 1, H, W)
            x_true_b = x_true.unsqueeze(0)           # (1, 1, H, W) on CPU

            x_hat = model(y_up_b).cpu().clamp(0.0, 1.0)

            psnr_val = metrics.PSNR(x_hat, x_true_b)
            ssim_val = metrics.SSIM(x_hat, x_true_b)

            unet_results[(s, sigma)]['psnr'].append(psnr_val)
            unet_results[(s, sigma)]['ssim'].append(ssim_val)
            unet_results[(s, sigma)]['x_hat'].append(x_hat.detach())

            print(f"  Image {idx+1:3d}/{len(test_ds)}  "
                  f"| PSNR={psnr_val:6.2f} dB  | SSIM={ssim_val:.4f}")

    mean_psnr = sum(unet_results[(s, sigma)]['psnr']) / len(unet_results[(s, sigma)]['psnr'])
    mean_ssim = sum(unet_results[(s, sigma)]['ssim']) / len(unet_results[(s, sigma)]['ssim'])
    unet_results[(s, sigma)]['mean_psnr'] = mean_psnr
    unet_results[(s, sigma)]['mean_ssim'] = mean_ssim
    print(f"\n  ➜  Mean PSNR = {mean_psnr:.2f} dB  |  Mean SSIM = {mean_ssim:.4f}")

## 3.8 Quantitative Results Summary

In [ ]:
print(f"\n{'─'*55}")
print(f"  {'Config':<22} {'Mean PSNR (dB)':>14} {'Mean SSIM':>12}")
print(f"{'─'*55}")
for (s, sigma), res in unet_results.items():
    label = f"s={s}, σ={sigma}"
    print(f"  {label:<22} {res['mean_psnr']:>14.2f} {res['mean_ssim']:>12.4f}")
print(f"{'─'*55}")

## 3.9 Qualitative Visualisation

For each scenario: degraded input (bilinear upsampled), U-Net reconstruction, ground truth.

In [ ]:
IMG_IDX = 0  # Which test image index to display

n_rows = len(unet_results)
fig, axes = plt.subplots(n_rows, 3, figsize=(13, 4 * n_rows))
if n_rows == 1:
    axes = [axes]

for row, ((s, sigma), res) in enumerate(unet_results.items()):
    # Rebuild a single sample for display
    test_ds = DegradedDataset(
        os.path.join(DATASET_PATH, 'test'), s, sigma, augment=False
    )
    y_up, x_true = test_ds[IMG_IDX]

    lr_disp  = y_up[0].numpy()
    hr_disp  = x_true[0].numpy()
    hat_disp = res['x_hat'][IMG_IDX][0, 0].numpy()

    psnr_val = res['psnr'][IMG_IDX]
    ssim_val = res['ssim'][IMG_IDX]

    ax_gt, ax_deg, ax_rec = axes[row]

    ax_gt.imshow(hr_disp,  cmap='gray', vmin=0, vmax=1)
    ax_gt.set_title('Ground Truth', fontsize=11); ax_gt.axis('off')

    ax_deg.imshow(lr_disp, cmap='gray', vmin=0, vmax=1)
    ax_deg.set_title(f'Degraded  (s={s}, σ={sigma})', fontsize=11); ax_deg.axis('off')

    ax_rec.imshow(hat_disp, cmap='gray', vmin=0, vmax=1)
    ax_rec.set_title(
        f'U-Net Reconstruction\nPSNR={psnr_val:.2f} dB  SSIM={ssim_val:.4f}',
        fontsize=11
    )
    ax_rec.axis('off')

plt.suptitle('U-Net (specialized per scenario) — Qualitative Results',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'unet_qualitative_results.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved to {OUTPUT_PATH}")

## 3.10 Save Results to Drive

Metrics CSV and reconstructed tensors are saved so Notebook 05 can load them for the final comparison.

In [ ]:
import csv

# ── Save metrics CSV ─────────────────────────────────────────────────────────
csv_path = os.path.join(OUTPUT_PATH, 'unet_metrics.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['scale_factor', 'noise_level', 'image_idx', 'psnr_db', 'ssim'])
    for (s, sigma), res in unet_results.items():
        for i, (p, ss) in enumerate(zip(res['psnr'], res['ssim'])):
            writer.writerow([s, sigma, i, round(p, 4), round(ss, 6)])

print(f"Metrics CSV saved → {csv_path}")

# ── Save reconstructed tensors ───────────────────────────────────────────────
for (s, sigma), res in unet_results.items():
    tag         = scenario_tag(s, sigma)
    tensors_dir = os.path.join(OUTPUT_PATH, tag)
    os.makedirs(tensors_dir, exist_ok=True)
    for i, x_hat in enumerate(res['x_hat']):
        torch.save(x_hat, os.path.join(tensors_dir, f'unet_recon_{i:03d}.pt'))

print(f"Reconstructed tensors saved → {OUTPUT_PATH}")
print("\n✅ Phase 3 complete. U-Net results are ready for comparison in Notebook 05.")